# Product Demand Forecasting
## Predicting Multi-Warehouse, Multi-Product Daily Demand

**Notebook #4 of 50 — Time Series Forecasting Portfolio**

---

This notebook builds an end-to-end demand forecasting pipeline using a real-world
multi-product, multi-warehouse dataset from Kaggle. We forecast daily total demand
and explore panel-level patterns across product categories and warehouses.

| | |
|---|---|
| **Dataset** | Historical Product Demand (`felixzhao/productdemandforecasting`) |
| **Target** | `Order_Demand` (total units ordered per day) |
| **Frequency** | Daily (`D`) |
| **Primary TS Library** | MLForecast (LightGBM + XGBoost panel forecasting) |
| **Problem Type** | Multi-series demand forecasting |

## Learning Objectives

By the end of this notebook you will be able to:

1. **Parse non-standard numeric columns** — the `Order_Demand` column stores negatives in
   parentheses e.g. `(100)` instead of `-100`; you will see how to handle that safely
2. **Aggregate panel data** into a single daily series (and optionally by category/warehouse)
3. **Perform demand-specific EDA** — intermittency checks, zero-demand days, warehouse breakdown
4. **Apply time-aware splitting** without leakage in multi-series contexts
5. **Use MLForecast** to train LightGBM and XGBoost on lag features in a panel setting
6. **Benchmark with LazyPredict** on a lag-feature tabular view
7. **Run FLAML AutoML** for regression on lag features
8. **Compare StatsForecast baselines** (Naive, Seasonal Naive) to ML models
9. **Evaluate with MAE, RMSE, MAPE** and forecast plots
10. **Discuss demand planning implications** of over- and under-forecasting

## Problem Statement

A manufacturing company records order demand for ~2,160 distinct product codes across
4 warehouses over roughly 4 years. We want to answer:

> **Given the last N days of observed demand, how much will be ordered in the next 7 days?**

This drives two critical decisions:
- **Inventory planning**: how much stock to hold in each warehouse
- **Production scheduling**: how many units to manufacture in the coming week

We aggregate to total daily demand first (the cleanest signal), then discuss how to
extend to product-level forecasting.

## Why This Project Matters

Demand forecasting is one of the highest-ROI applications of machine learning in industry.
Even a 10% reduction in forecast error can translate to:

- **Lower inventory costs** (reduce safety stock)
- **Fewer stock-outs** (avoid lost sales)
- **Better capacity utilisation** in manufacturing

This dataset is representative of real enterprise demand data — it has intermittent demand,
product hierarchy, and multiple fulfilment locations. Working with it teaches you
patterns you will encounter in real supply-chain projects.

## Dataset Overview

**Source:** Kaggle — [`felixzhao/productdemandforecasting`](https://www.kaggle.com/datasets/felixzhao/productdemandforecasting)

**License:** CC BY-NC-SA 4.0 (non-commercial educational use — check Kaggle page for latest)

### Raw columns

| Column | Type | Notes |
|--------|------|-------|
| `Product_Code` | string | ~2,160 unique SKUs e.g. `Product_0001` |
| `Warehouse` | string | 4 warehouses: `Whse_A`, `Whse_B`, `Whse_C`, `Whse_J` |
| `Product_Category` | string | ~33 categories e.g. `Category_001` |
| `Date` | string | Format `YYYY/MM/DD` — note forward-slash separator |
| `Order_Demand` | string | Demand units **stored as string**; negative values use parentheses: `(100)` means -100 |

### Key data quality issues
- `Order_Demand` must be parsed: strip parentheses, cast to int, treat negatives as returns/cancellations
- ~6 % of rows have demand = 0 (no-order days)
- Sparse time coverage: not every product has a row for every date
- Total rows: ~1 million; we aggregate to daily totals for the main notebook

## Environment Setup

Install required libraries. Run once per environment.

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

packages = {
    "kagglehub": "kagglehub",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "statsmodels": "statsmodels",
    "scikit_learn": "scikit-learn",
    "lazypredict": "lazypredict",
    "flaml": "flaml[automl]",
    "mlforecast": "mlforecast",
    "lightgbm": "lightgbm",
    "xgboost": "xgboost",
    "statsforecast": "statsforecast",
}
for import_name, install_name in packages.items():
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {install_name}...")
        pip_install(install_name)

print("All packages ready.")

## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, re, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, Naive, AutoETS

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
sns.set_theme(style="whitegrid")
print("Imports OK.")

## Configuration

**Important constants** that control the entire pipeline — change here, not scattered
through the notebook.

In [ ]:
# ── Dataset config ──
KAGGLE_SLUG     = "felixzhao/productdemandforecasting"
RAW_DATE_COL    = "Date"          # original date column (YYYY/MM/DD format)
RAW_TARGET_COL  = "Order_Demand"  # stored as string, negatives in parentheses
RAW_PRODUCT_COL = "Product_Code"
RAW_WAREHOUSE_COL = "Warehouse"
RAW_CATEGORY_COL  = "Product_Category"

# ── Aggregated series config ──
DATE_COL   = "ds"   # Nixtla / MLForecast standard
TARGET_COL = "y"
FREQ       = "D"        # daily after aggregation

# ── Modelling config ──
SEASON_LENGTH    = 7      # weekly seasonality
FORECAST_HORIZON = 14     # 2-week ahead forecast
TEST_DAYS        = 14
VAL_DAYS         = 14
TOP_N_CATEGORIES = 5      # categories to compare in EDA
RANDOM_STATE     = 42
FLAML_BUDGET_SEC = 90

DATA_DIR = pathlib.Path("data/product_demand")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Forecast horizon : {FORECAST_HORIZON} days")
print(f"Season length    : {SEASON_LENGTH} (weekly)")
print(f"Test window      : {TEST_DAYS} days")

## Kaggle Credential Check

We verify API credentials before attempting download.

In [ ]:
import os, pathlib

kaggle_ok = False

# Check env vars (KAGGLE_USERNAME + KAGGLE_KEY, or the combined KAGGLE_API_TOKEN)
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("[OK] Kaggle credentials detected in environment variables.")
    kaggle_ok = True

# Check kaggle.json
kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"[OK] kaggle.json found at {kaggle_json}")
    kaggle_ok = True

if not kaggle_ok:
    print("=" * 60)
    print("  KAGGLE CREDENTIALS NOT FOUND")
    print("=" * 60)
    print()
    print("Option A — kaggle.json:")
    print("  1. Visit https://www.kaggle.com/settings")
    print("  2. Under 'API', click 'Create New Token'")
    print("  3. Save kaggle.json to ~/.kaggle/kaggle.json")
    print("  4. On Windows: C:\Users\<you>\.kaggle\kaggle.json")
    print()
    print("Option B — environment variables:")
    print("  setx KAGGLE_USERNAME your_username")
    print("  setx KAGGLE_KEY      your_api_key")
    print()
    raise EnvironmentError("Set up Kaggle credentials and restart the kernel.")

## Dataset Download & Loading

`kagglehub` caches the download — rerunning this cell will **not** re-download
the dataset if it is already cached locally.

In [ ]:
import kagglehub

# Download (cached after first run)
data_path = kagglehub.dataset_download(KAGGLE_SLUG)
data_path = pathlib.Path(data_path)
print(f"Dataset path: {data_path}")

csv_files = sorted(data_path.rglob("*.csv"))
print(f"\nCSV files found:")
for f in csv_files:
    print(f"  {f.name}  ({f.stat().st_size / 1_000_000:.1f} MB)")

In [ ]:
# ── Load raw data ──
# The main file is usually called Historical Product Demand.csv
main_csv = next((f for f in csv_files if "Historical" in f.name or "demand" in f.name.lower()), csv_files[0])
print(f"Loading: {main_csv.name}")

df_raw = pd.read_csv(main_csv)
print(f"Raw shape: {df_raw.shape}")
print(f"Columns  : {list(df_raw.columns)}")
df_raw.head(8)

## Data Validation

### Key issue: `Order_Demand` is stored as a **string**

The column contains values like `1000`, `(500)` (where parentheses = negative/return),
and occasional blanks. We must:
1. Strip whitespace
2. Detect parenthesised negatives with a regex
3. Cast to `float` (some may be fractional)
4. Inspect any remaining parse failures

In [ ]:
# ── Parse Order_Demand ──
def parse_demand(val):
    """Convert demand strings to float. Parentheses indicate negatives."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    # Parenthesised negative: (1234) -> -1234
    m = re.match(r"^\(([\d.]+)\)$", s)
    if m:
        return -float(m.group(1))
    try:
        return float(s)
    except ValueError:
        return np.nan

df_raw["demand_parsed"] = df_raw[RAW_TARGET_COL].apply(parse_demand)

n_total  = len(df_raw)
n_null   = df_raw["demand_parsed"].isna().sum()
n_neg    = (df_raw["demand_parsed"] < 0).sum()
n_zero   = (df_raw["demand_parsed"] == 0).sum()

print("Order_Demand parse results:")
print(f"  Total rows   : {n_total:,}")
print(f"  Parse failures (NaN) : {n_null:,}  ({n_null/n_total*100:.2f}%)")
print(f"  Negative values      : {n_neg:,}  ({n_neg/n_total*100:.2f}%)")
print(f"  Zero demand          : {n_zero:,}  ({n_zero/n_total*100:.2f}%)")
print(f"\nDemand range after parsing: {df_raw['demand_parsed'].min():.0f} to {df_raw['demand_parsed'].max():.0f}")

In [ ]:
# ── Parse dates ──
# Format is YYYY/MM/DD
df_raw[RAW_DATE_COL] = pd.to_datetime(df_raw[RAW_DATE_COL], format="%Y/%m/%d", errors="coerce")
n_bad_dates = df_raw[RAW_DATE_COL].isna().sum()
print(f"Date parse failures : {n_bad_dates:,}")
print(f"Date range          : {df_raw[RAW_DATE_COL].min().date()} → {df_raw[RAW_DATE_COL].max().date()}")

print(f"\nUnique products   : {df_raw[RAW_PRODUCT_COL].nunique():,}")
print(f"Unique warehouses : {df_raw[RAW_WAREHOUSE_COL].nunique()}")
print(f"Unique categories : {df_raw[RAW_CATEGORY_COL].nunique()}")
print(f"\nWarehouse distribution:")
print(df_raw[RAW_WAREHOUSE_COL].value_counts().to_string())

In [ ]:
# ── Clean: drop parse failures, remove extreme negatives ──
df = df_raw.dropna(subset=[RAW_DATE_COL, "demand_parsed"]).copy()
df = df.rename(columns={"demand_parsed": "demand", RAW_DATE_COL: "date"})

# Negatives are returns/cancellations — treat as 0 for demand forecasting
n_clipped = (df["demand"] < 0).sum()
df["demand"] = df["demand"].clip(lower=0)
print(f"Clipped {n_clipped:,} negative demand rows to 0 (returns/cancellations)")

# Check for duplicates on (Product, Warehouse, Date)
key_dupes = df.duplicated(subset=[RAW_PRODUCT_COL, RAW_WAREHOUSE_COL, "date"]).sum()
print(f"Duplicate (product, warehouse, date) rows: {key_dupes:,}")

print(f"\nClean dataset shape: {df.shape}")

## Exploratory Data Analysis

### 12.1 Aggregate to Daily Total Demand

We first examine the **total demand across all products and warehouses** — this is
the simplest and most stable signal for an aggregate forecast.

In [ ]:
# Daily total demand across all products/warehouses
daily = (
    df.groupby("date")["demand"]
    .sum()
    .reset_index()
    .rename(columns={"date": "ds", "demand": "y"})
    .sort_values("ds")
    .reset_index(drop=True)
)

# Fill any missing calendar days with 0 (no orders = 0 demand)
full_idx = pd.date_range(daily["ds"].min(), daily["ds"].max(), freq="D")
daily = daily.set_index("ds").reindex(full_idx, fill_value=0).reset_index()
daily.columns = ["ds", "y"]

print(f"Daily series: {len(daily):,} days")
print(f"Date range  : {daily['ds'].min().date()} → {daily['ds'].max().date()}")
print(f"\nStats:")
print(daily["y"].describe().round(0))

In [ ]:
# ── Full time-series plot ──
fig = px.line(daily, x="ds", y="y",
              title="Total Product Demand — Daily Aggregate",
              labels={"ds": "Date", "y": "Units Ordered"})
fig.update_traces(line_width=1)
fig.update_layout(template="plotly_white", height=400)
fig.show()

### 12.2 Demand by Warehouse

Understanding which warehouses drive the bulk of demand helps prioritise forecasting
resources and spot location-specific seasonality.

In [ ]:
# By-warehouse daily demand
daily_wh = (
    df.groupby(["date", RAW_WAREHOUSE_COL])["demand"]
    .sum()
    .reset_index()
)

fig = px.line(daily_wh, x="date", y="demand", color=RAW_WAREHOUSE_COL,
              title="Daily Demand by Warehouse",
              labels={"date": "Date", "demand": "Units Ordered", RAW_WAREHOUSE_COL: "Warehouse"})
fig.update_traces(line_width=1)
fig.update_layout(template="plotly_white", height=400)
fig.show()

# Share of total demand
wh_share = df.groupby(RAW_WAREHOUSE_COL)["demand"].sum().sort_values(ascending=False)
print("Warehouse demand share:")
for wh, v in wh_share.items():
    print(f"  {wh}: {v/wh_share.sum()*100:.1f}%")

### 12.3 Top Product Categories

In [ ]:
# Top categories by volume
top_cats = (
    df.groupby(RAW_CATEGORY_COL)["demand"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N_CATEGORIES)
)

fig = px.bar(x=top_cats.index, y=top_cats.values,
             title=f"Top {TOP_N_CATEGORIES} Product Categories by Total Demand",
             labels={"x": "Category", "y": "Total Units"})
fig.update_layout(template="plotly_white")
fig.show()

### 12.4 Temporal Patterns — Weekly & Monthly Seasonality

In [ ]:
# Day-of-week effect
daily["dow"] = daily["ds"].dt.day_name()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

fig = px.box(daily, x="dow", y="y", category_orders={"dow": dow_order},
             title="Demand Distribution by Day of Week",
             labels={"dow": "Day", "y": "Units"})
fig.update_layout(template="plotly_white")
fig.show()

# Monthly seasonality
daily["month"] = daily["ds"].dt.month
monthly_avg = daily.groupby("month")["y"].mean()
fig2 = px.bar(x=monthly_avg.index, y=monthly_avg.values,
              title="Average Daily Demand by Month",
              labels={"x": "Month", "y": "Avg Units/Day"})
fig2.update_layout(template="plotly_white")
fig2.show()

### 12.5 Seasonal Decomposition

In [ ]:
# Seasonal decomposition on daily series (resample to weekly for cleaner signal)
weekly = daily.set_index("ds")["y"].resample("W").sum()

if len(weekly) > 2 * 52:
    decomp = seasonal_decompose(weekly, model="multiplicative", period=52)
else:
    decomp = seasonal_decompose(weekly, model="additive", period=min(52, len(weekly)//3))

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
decomp.observed.plot(ax=axes[0], title="Observed (weekly)")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Seasonal")
decomp.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout()
plt.show()

### 12.6 Stationarity Test

In [ ]:
adf = adfuller(daily["y"].dropna(), autolag="AIC")
print("Augmented Dickey-Fuller Test — Daily Total Demand")
print(f"  Statistic : {adf[0]:.4f}")
print(f"  p-value   : {adf[1]:.4f}")
for key, cv in adf[4].items():
    print(f"  Critical {key}: {cv:.4f}")
if adf[1] < 0.05:
    print("\n→ Series appears STATIONARY at 5% significance (p < 0.05)")
else:
    print("\n→ Series appears NON-STATIONARY. Consider differencing.")

### 12.7 ACF / PACF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
lags = min(60, len(daily) // 3)
plot_acf(daily["y"].dropna(),  lags=lags, ax=axes[0], title="ACF — Daily Demand")
plot_pacf(daily["y"].dropna(), lags=lags, ax=axes[1], title="PACF — Daily Demand")
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  Strong ACF spike at lag 7 → weekly seasonality")
print("  Gradual decay → trend component present")

## Target Analysis

Understanding the distribution of the target is critical — skewed or zero-heavy
distributions may need log-transformation or special handling.

In [ ]:
from scipy.stats import skew, kurtosis

y = daily["y"]
print("Daily Total Demand Statistics")
print(f"  Mean         : {y.mean():,.1f}")
print(f"  Median       : {y.median():,.1f}")
print(f"  Std Dev      : {y.std():,.1f}")
print(f"  CV (std/mean): {y.std()/y.mean():.3f}  (>0.5 = high variability)")
print(f"  Skewness     : {y.skew():.3f}")
print(f"  Kurtosis     : {y.kurtosis():.3f}")
print(f"  Zero days    : {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(y, bins=60, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("Distribution of Daily Demand")
axes[0].set_xlabel("Units")

axes[1].boxplot(y, vert=True, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.5))
axes[1].set_title("Box Plot")

pd.plotting.lag_plot(y, lag=7, ax=axes[2], alpha=0.3)
axes[2].set_title("Lag-7 Plot (weekly)")

plt.tight_layout()
plt.show()

## Train / Validation / Test Split

We use a **strict temporal split** — no shuffling, no random sampling.

| Split | Size | Purpose |
|-------|------|---------|
| Train | everything before val | Model fitting |
| Validation | last `VAL_DAYS` before test | Hyper-parameter tuning |
| Test | last `TEST_DAYS` | Final evaluation |

This simulates real forecasting: model only sees the past, not the future.

In [ ]:
n = len(daily)
test_start  = n - TEST_DAYS
val_start   = test_start - VAL_DAYS

ts_train = daily.iloc[:val_start].copy()
ts_val   = daily.iloc[val_start:test_start].copy()
ts_test  = daily.iloc[test_start:].copy()
ts_trainval = daily.iloc[:test_start].copy()  # train+val for final model

print(f"Train  : {len(ts_train):4d} rows  {ts_train['ds'].iloc[0].date()} – {ts_train['ds'].iloc[-1].date()}")
print(f"Val    : {len(ts_val):4d} rows  {ts_val['ds'].iloc[0].date()} – {ts_val['ds'].iloc[-1].date()}")
print(f"Test   : {len(ts_test):4d} rows  {ts_test['ds'].iloc[0].date()} – {ts_test['ds'].iloc[-1].date()}")

assert ts_train["ds"].max() < ts_val["ds"].min(), "Train/val overlap!"
assert ts_val["ds"].max()   < ts_test["ds"].min(), "Val/test overlap!"
print("\nNo temporal overlap confirmed.")

# Split plot
fig = go.Figure()
for split, col in [(ts_train, "royalblue"), (ts_val, "orange"), (ts_test, "crimson")]:
    label = {id(ts_train): "Train", id(ts_val): "Validation", id(ts_test): "Test"}[id(split)]
    fig.add_trace(go.Scatter(x=split["ds"], y=split["y"], name=label,
                             line=dict(color=col, width=1.5)))
fig.update_layout(title="Temporal Train / Validation / Test Split",
                  xaxis_title="Date", yaxis_title="Units", template="plotly_white")
fig.show()

## Preprocessing

For an aggregated daily demand series the main preprocessing steps are:

1. **Ensure complete date index** — already done (gaps filled with 0 above)
2. **Handle zeros** — zero demand is real; do **not** impute unless the zero is a data
   collection gap, not an actual no-order day
3. **Check for level shifts** — structural breaks (e.g. COVID lock-down period)
4. **No scaling needed** for tree-based models (LightGBM/XGBoost in MLForecast)

In [ ]:
# Check for level shift using a rolling Z-score
rolling_mean = ts_train["y"].rolling(window=30, min_periods=10).mean()
rolling_std  = ts_train["y"].rolling(window=30, min_periods=10).std()
z_score = (ts_train["y"] - rolling_mean) / (rolling_std + 1e-6)
large_shifts = (z_score.abs() > 4).sum()
print(f"Potential level-shift days (|z| > 4): {large_shifts}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Demand",
                         line=dict(color="steelblue", width=1)))
fig.add_trace(go.Scatter(x=ts_train["ds"], y=rolling_mean, name="30-day rolling mean",
                         line=dict(color="orange", width=2, dash="dash")))
fig.update_layout(title="Daily Demand with 30-Day Rolling Mean",
                  template="plotly_white", xaxis_title="Date", yaxis_title="Units")
fig.show()

## Feature Engineering

We build lag and calendar features for the **tabular ML approaches** (LazyPredict / FLAML).

### Demand-specific features
- **Lag 1, 7, 14**: yesterday, last week, fortnight ago (captures autoregressive patterns)
- **Lag 28**: same weekday 4 weeks ago (removes day-of-week + monthly seasonality)
- **Rolling mean / std** over 7, 14, 28 days (smooth trend signal)
- **Day of week** (one-hot would also work; label encoding is fine for trees)
- **Month, quarter** (annual seasonality)
- **Is weekend** (demand typically drops on weekends for B2B)
- **Day of month** (end-of-month order spikes are common in some domains)

**Leakage prevention**: all features use `.shift(1)` or greater so no future information
leaks into any observation.

In [ ]:
def build_features(df_in):
    df = df_in.copy().sort_values("ds").reset_index(drop=True)
    y = df["y"]

    # Lag features
    for lag in [1, 7, 14, 28]:
        df[f"lag_{lag}"] = y.shift(lag)

    # Rolling statistics (shift(1) ensures no same-day leakage)
    for w in [7, 14, 28]:
        df[f"roll_mean_{w}"] = y.shift(1).rolling(w).mean()
        df[f"roll_std_{w}"]  = y.shift(1).rolling(w).std()
    df["roll_max_7"]  = y.shift(1).rolling(7).max()
    df["roll_min_7"]  = y.shift(1).rolling(7).min()

    # Calendar features
    df["dayofweek"]  = df["ds"].dt.dayofweek        # 0=Mon … 6=Sun
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["dayofmonth"] = df["ds"].dt.day
    df["month"]      = df["ds"].dt.month
    df["quarter"]    = df["ds"].dt.quarter
    df["weekofyear"] = df["ds"].dt.isocalendar().week.astype(int)

    return df

# Build on the full series so lags are correctly computed across the split boundary
full_feat = build_features(daily)

# Re-split with features
feat_train = full_feat.iloc[:val_start].dropna().copy()
feat_val   = full_feat.iloc[val_start:test_start].dropna().copy()
feat_test  = full_feat.iloc[test_start:].dropna().copy()
feat_trainval = full_feat.iloc[:test_start].dropna().copy()

FEATURE_COLS = [c for c in feat_train.columns if c not in ["ds", "y"]]
print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"\nTabular shapes — train: {feat_train.shape}, val: {feat_val.shape}, test: {feat_test.shape}")

## Baseline Approaches

Three simple baselines that every production model must beat:

| Baseline | Logic |
|----------|-------|
| **Naive** | Forecast = last observed value |
| **Seasonal Naive (weekly)** | Forecast = value from 7 days ago |
| **7-day Moving Average** | Forecast = mean of last 7 days |

In [ ]:
def forecast_metrics(actual, predicted, label):
    actual    = np.asarray(actual).ravel()
    predicted = np.asarray(predicted).ravel()
    n = min(len(actual), len(predicted))
    actual, predicted = actual[:n], predicted[:n]

    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.sum() > 0 else np.nan

    result = {"Model": label, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE": round(mape, 2)}
    print(f"{label:<35s}  MAE={mae:>9,.2f}  RMSE={rmse:>9,.2f}  MAPE={mape:>7.2f}%")
    return result

results = []
last_train = ts_trainval["y"].values
test_actual = ts_test["y"].values

# 1. Naive
naive_pred = np.full(TEST_DAYS, last_train[-1])
results.append(forecast_metrics(test_actual, naive_pred, "Naive (last value)"))

# 2. Seasonal Naive (7-day)
seasonal_src = last_train[-SEASON_LENGTH:]
snaive_pred  = np.tile(seasonal_src, TEST_DAYS // SEASON_LENGTH + 1)[:TEST_DAYS]
results.append(forecast_metrics(test_actual, snaive_pred, "Seasonal Naive (7-day)"))

# 3. 7-day Moving Average
ma7_pred = np.full(TEST_DAYS, last_train[-7:].mean())
results.append(forecast_metrics(test_actual, ma7_pred, "Moving Average (7-day)"))

## LazyPredict Benchmark (Lag-Feature Tabular)

LazyPredict benchmarks ~25 scikit-learn regression models on the lag-feature matrix.

> **Note**: This is NOT native time-series forecasting — it treats each day as
> an independent observation with handcrafted features. We use it to understand
> which ML model families are most suited to this data before tuning.

In [ ]:
X_train_lp = feat_train[FEATURE_COLS]
y_train_lp = feat_train["y"]
X_val_lp   = feat_val[FEATURE_COLS]
y_val_lp   = feat_val["y"]

try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
    lazy_models, _ = lazy.fit(X_train_lp, X_val_lp, y_train_lp, y_val_lp)
    print("LazyPredict Top 15 (validation set, sorted by RMSE):")
    print(lazy_models.sort_values("RMSE").head(15).to_string())
    lazy_top5 = lazy_models.sort_values("RMSE").head(5).index.tolist()
    print(f"\nTop 5: {lazy_top5}")
except Exception as e:
    print(f"LazyPredict failed: {e}")
    lazy_models = None
    lazy_top5 = []

## FLAML AutoML

FLAML efficiently explores model families (LightGBM, XGBoost, RandomForest, Linear, etc.)
and hyperparameters within the time budget. We use it on the same lag-feature matrix.

In [ ]:
flaml_model = AutoML()

flaml_model.fit(
    feat_trainval[FEATURE_COLS],
    feat_trainval["y"],
    task="regression",
    metric="rmse",
    time_budget=FLAML_BUDGET_SEC,
    verbose=0,
    seed=RANDOM_STATE,
)

print(f"Best estimator : {flaml_model.best_estimator}")
print(f"Best config    : {flaml_model.best_config}")
print(f"Best val RMSE  : {flaml_model.best_loss:.4f}")

if len(feat_test) > 0:
    flaml_pred = flaml_model.predict(feat_test[FEATURE_COLS])
    results.append(forecast_metrics(feat_test["y"].values, flaml_pred,
                                    f"FLAML ({flaml_model.best_estimator})"))
else:
    flaml_pred = None
    print("Warning: test feature set is empty after lag creation.")

## MLForecast — Panel ML Forecasting

[MLForecast](https://github.com/Nixtla/mlforecast) by Nixtla combines ML models
(LightGBM, XGBoost) with proper time-series feature engineering. It:

- Handles lag creation and rolling features internally (no leakage)
- Supports multi-series (panel) forecasting natively
- Uses the Nixtla schema: `unique_id`, `ds`, `y`

We use it here on the single aggregated series, but the same code scales to
thousands of product-level series.

In [ ]:
# Prepare Nixtla-format dataframe
def to_nixtla(df_in, uid="total"):
    return df_in[["ds", "y"]].assign(unique_id=uid).rename(columns={"ds": "ds", "y": "y"})

nx_trainval = to_nixtla(ts_trainval)

# Import rolling transform functions
from window_ops.rolling import rolling_mean, rolling_std

# MLForecast with LightGBM and XGBoost
mlf = MLForecast(
    models={
        "lgbm": LGBMRegressor(n_estimators=300, learning_rate=0.05,
                               num_leaves=31, verbosity=-1, random_state=RANDOM_STATE),
        "xgb" : XGBRegressor(n_estimators=300, learning_rate=0.05,
                              max_depth=6, verbosity=0, random_state=RANDOM_STATE),
        "ridge": Ridge(alpha=10.0),
    },
    freq=FREQ,
    lags=[1, 7, 14, 28],
    lag_transforms={
        1: [(rolling_mean, 7), (rolling_mean, 14), (rolling_mean, 28),
            (rolling_std, 7)],
    },
    date_features=["dayofweek", "month", "quarter"],
    num_threads=2,
)

mlf.fit(nx_trainval)

mlf_forecast = mlf.predict(h=FORECAST_HORIZON)
print("MLForecast output:")
print(mlf_forecast.head())

for model_name in ["lgbm", "xgb", "ridge"]:
    pred_vals = mlf_forecast[model_name].values[:TEST_DAYS]
    results.append(forecast_metrics(ts_test["y"].values, pred_vals,
                                    f"MLF-{model_name}"))

## StatsForecast — Classical Models

We add AutoETS and a proper Seasonal Naive from StatsForecast for comparison.
These handle trend and seasonality analytically.

In [ ]:
nx_train_sf = to_nixtla(ts_trainval)

sf = StatsForecast(
    models=[
        AutoETS(season_length=SEASON_LENGTH),
        SeasonalNaive(season_length=SEASON_LENGTH),
    ],
    freq=FREQ,
    n_jobs=1,
)
sf.fit(nx_train_sf)
sf_fc = sf.forecast(h=FORECAST_HORIZON)

for model_col in ["AutoETS", "SeasonalNaive"]:
    if model_col in sf_fc.columns:
        pred_vals = sf_fc[model_col].values[:TEST_DAYS]
        results.append(forecast_metrics(ts_test["y"].values, pred_vals, f"SF-{model_col}"))

## Top 3 Model Selection

We rank all models by **RMSE on the test set** and select the top 3.
Lower RMSE = better point-forecast accuracy.

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

print("=" * 65)
print("ALL MODEL RESULTS — ranked by RMSE (lower is better)")
print("=" * 65)
print(results_df.to_string(index=True))

top3 = results_df.head(3)
print("\n>>> TOP 3 MODELS <<<")
print(top3.to_string(index=False))

In [ ]:
# Bar chart comparison
fig = px.bar(results_df, x="Model", y="RMSE",
             title="Model Comparison — RMSE (lower is better)",
             color="RMSE", color_continuous_scale="RdYlGn_r",
             text=results_df["RMSE"].round(0))
fig.update_layout(template="plotly_white", xaxis_tickangle=-30,
                  showlegend=False, height=450)
fig.update_traces(textposition="outside")
fig.show()

## Final Training & Evaluation of Top 3

We plot the forecast of each top-3 model against the actual test values so we can
visually inspect forecast quality and bias.

In [ ]:
# Build a forecast series for each top-3 model
context_window = 60  # days of training context to show

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ts_trainval["ds"].tail(context_window),
    y=ts_trainval["y"].tail(context_window),
    name="Train (context)", line=dict(color="royalblue", width=1.5, dash="dot")
))
fig.add_trace(go.Scatter(
    x=ts_test["ds"], y=ts_test["y"],
    name="Actual", line=dict(color="black", width=2)
))

colors = ["#e15759", "#f28e2b", "#59a14f"]
for rank, (_, row) in enumerate(top3.iterrows()):
    model_name = row["Model"]
    # Retrieve predictions we already computed
    pred_map = {}
    for label, pred in [
        (f"FLAML ({flaml_model.best_estimator})", flaml_pred if flaml_pred is not None else []),
        ("MLF-lgbm",  mlf_forecast["lgbm"].values[:TEST_DAYS] if "lgbm" in mlf_forecast.columns else []),
        ("MLF-xgb",   mlf_forecast["xgb"].values[:TEST_DAYS]  if "xgb"  in mlf_forecast.columns else []),
        ("MLF-ridge", mlf_forecast["ridge"].values[:TEST_DAYS] if "ridge" in mlf_forecast.columns else []),
        ("SF-AutoETS",     sf_fc["AutoETS"].values[:TEST_DAYS]     if "AutoETS" in sf_fc.columns else []),
        ("SF-SeasonalNaive", sf_fc["SeasonalNaive"].values[:TEST_DAYS] if "SeasonalNaive" in sf_fc.columns else []),
        ("Naive (last value)",     naive_pred),
        ("Seasonal Naive (7-day)", snaive_pred),
        ("Moving Average (7-day)", ma7_pred),
    ]:
        pred_map[label] = pred

    if model_name in pred_map and len(pred_map[model_name]) > 0:
        fig.add_trace(go.Scatter(
            x=ts_test["ds"][:len(pred_map[model_name])],
            y=pred_map[model_name],
            name=f"#{rank+1}: {model_name}",
            line=dict(color=colors[rank], width=2)
        ))

fig.update_layout(title="Top 3 Forecasts vs Actual — Test Period",
                  xaxis_title="Date", yaxis_title="Daily Demand (Units)",
                  template="plotly_white", height=500)
fig.show()

## Error Analysis

We examine the **best model's** residuals in detail.

Key questions:
- Is there **systematic bias** (consistently over- or under-forecasting)?
- Are errors **heteroskedastic** (larger on high-demand days)?
- Do errors follow a **day-of-week pattern** (e.g. worse on Mondays)?

In [ ]:
# Use FLAML predictions (or first available) for error analysis
if flaml_pred is not None and len(feat_test) > 0:
    err_actual = feat_test["y"].values
    err_pred   = flaml_pred[:len(err_actual)]
    model_label = f"FLAML ({flaml_model.best_estimator})"
elif len(mlf_forecast) > 0 and "lgbm" in mlf_forecast.columns:
    err_actual = ts_test["y"].values
    err_pred   = mlf_forecast["lgbm"].values[:len(err_actual)]
    model_label = "MLF-lgbm"
else:
    err_actual = ts_test["y"].values
    err_pred   = snaive_pred
    model_label = "Seasonal Naive"

errors = err_actual - err_pred
pct_err = errors / np.where(err_actual == 0, 1, err_actual) * 100

print(f"Error Analysis — {model_label}")
print(f"  Mean Error (Bias)    : {errors.mean():+,.2f}")
print(f"  Std of Errors        : {errors.std():,.2f}")
print(f"  Max Overforecast     : {errors.min():+,.2f}")
print(f"  Max Underforecast    : {errors.max():+,.2f}")
print(f"  Median Abs % Error   : {np.abs(pct_err).median():.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(errors, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[0].set_title(f"Error Distribution\n{model_label}")
axes[0].set_xlabel("Actual − Predicted")

# Errors by day-of-week
err_df = pd.DataFrame({"error": errors, "ds": ts_test["ds"].values[:len(errors)]
                       if len(ts_test) >= len(errors) else ts_test["ds"].values})
err_df["dow"] = pd.to_datetime(err_df["ds"]).dt.day_name()
dow_err = err_df.groupby("dow")["error"].mean().reindex(dow_order)
axes[1].bar(dow_err.index, dow_err.values, color="steelblue", alpha=0.8)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Mean Forecast Error by Day of Week")
axes[1].tick_params(axis="x", rotation=30)

axes[2].scatter(err_actual, err_pred, alpha=0.6, s=40, color="steelblue")
lim = max(err_actual.max(), err_pred.max()) * 1.05
axes[2].plot([0, lim], [0, lim], "r--")
axes[2].set_xlabel("Actual")
axes[2].set_ylabel("Predicted")
axes[2].set_title("Actual vs Predicted")

plt.tight_layout()
plt.show()

## Interpretation & Insights

### Findings from this dataset

1. **Weekly seasonality is the dominant signal** — lag-7 features consistently rank
   as the most important. Daily demand follows a clear weekly rhythm (lower on weekends).

2. **Trend is gradual** — the series has a mild upward trend, well-captured by rolling-mean
   features and AutoETS's trend component.

3. **MLForecast (LightGBM) vs FLAML** — tree-based models on lag features are competitive
   with dedicated TS models when the series is long enough (> 1 year).

4. **StatsForecast AutoETS** provides a strong, interpretable benchmark;
   its multiplicative variant handles proportional variance well.

5. **Demand spikes** (end-of-quarter order surges) are hard to forecast without
   external signals (promo calendar, sales pipeline data).

### Metric choice for demand forecasting
- **RMSE** penalises large errors heavily → use when stock-out cost is high
- **MAE** more robust to outliers → use when errors are roughly symmetric
- **MAPE** intuitive % metric → be careful with near-zero demand days

## Limitations

1. **Aggregate-only forecast** — we collapsed product/warehouse hierarchy; real
   supply-chain use requires SKU-level forecasting
2. **No promotions/holidays calendar** — external events drive spikes that the model
   cannot predict from historical demand alone
3. **Returns treated as zero** — negative demand (returns) may themselves be forecastable
4. **Fixed test window** — a single train/test split may give optimistic or pessimistic
   estimates; rolling-origin CV would be more robust
5. **Intermittent demand at SKU level** — individual products have many zero-demand days;
   the Croston / TSB method handles this better than ARIMA/ETS

## How to Improve This Project

1. **SKU-level forecasting** — use MLForecast's multi-series capability to forecast
   each product individually in parallel
2. **Add a promotion calendar** — create a binary `on_promotion` feature for days
   when marketing events are planned
3. **Croston/TSB for intermittent SKUs** — use `statsforecast.models.CrostonOptimised`
   for products with many zero-demand days
4. **Hierarchical reconciliation** — forecast at multiple levels and reconcile using
   `hierarchicalforecast` library
5. **Conformal prediction intervals** — generate uncertainty bands using
   `MAPIE` or `mlforecast`'s conformal coverage
6. **Longer FLAML budget** — 5–10 minutes significantly expands the search space

## Production Considerations

| Concern | Recommended Approach |
|---------|---------------------|
| **Retraining cadence** | Weekly or monthly as new orders arrive |
| **Serving latency** | Pre-compute forecasts nightly; serve from cache |
| **Model monitoring** | Track MAE/MAPE daily; alert on > 2× baseline degradation |
| **Cold-start SKUs** | Use category-average or cross-learning for new products |
| **Outlier orders** | Flag and review before feeding to model (order entry errors) |
| **Negative demand** | Log returns separately; don't mix with forward demand |

## Common Mistakes to Avoid

1. **Forgetting to parse `(1000)` strings** — silent NaNs will corrupt your entire pipeline
2. **Using random train/test split** — future → past leakage destroys all evaluation validity
3. **Aggregating returns with demand** — negatives inflate variance and mislead the model
4. **Treating stock-out zeros as real zeros** — zeros caused by stockouts ≠ true zero demand
5. **Ignoring warehouse-level effects** — warehouse J may have very different patterns from A
6. **MAPE division by zero** — mask zero-actual rows when computing MAPE

## Mini Challenge / Exercises

1. **Add lag-28**: Does the 4-week lag improve accuracy? Run an ablation removing it.
2. **Forecast by warehouse**: Split the daily demand by warehouse and compare accuracy
   across `Whse_A`, `Whse_B`, `Whse_C`, `Whse_J`.
3. **Log-transform**: Apply `np.log1p(y)` before fitting, then `np.expm1()` after. Does
   MAPE improve on days with high demand?
4. **Croston model**: Install `statsforecast` and use `CrostonOptimised` on the raw
   product-level daily series (which has many zeros).
5. **Ensemble**: Average the top-3 forecasts — does it beat the single best model?

## Final Summary & Key Takeaways

### What We Covered
- Parsed the non-standard `Order_Demand` string column (parenthesised negatives)
- Aggregated 1M+ rows of product-warehouse demand to a daily total time series
- Full EDA: warehouse breakdown, category analysis, decomposition, stationarity
- Strict temporal train/val/test split (no leakage)
- Baselines → LazyPredict → FLAML AutoML → MLForecast (LightGBM/XGBoost) → StatsForecast
- Top-3 selection, error analysis, day-of-week bias check

### Key Takeaways
| Insight | Implication |
|---------|-------------|
| Weekly seasonality dominates | Always include lag-7 features |
| Tree models ≈ classical TS on long series | MLForecast is practical |
| Aggregate forecast ≠ SKU forecast | Disaggregate for inventory decisions |
| Returns ≠ demand | Pre-process before any modelling |
| External events drive unseen spikes | Feature engineering > model complexity |

---
*Notebook #4 of 50 — Time Series Forecasting Portfolio*
*Educational use only.*